In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

RAW_DIR = "sources/nube_2023"
OUT_FILE = "data/nube_manzana_2023.laz"
GPKG = "data/catastro_manzana.gpkg"

# Classified point cloud of the Ayuntamiento de Madrid, 2023, sheets of 1 x 1 km:
BASE_URL = "https://geoportal.madrid.es/fsdescargas/IDEAM_WBGEOPORTAL/ELEVACIONES/2023/NUBE_PUNTOS/LAS_clasificado/"
SHEETS = ["439-4474", "440-4474"]

# Margin around the block, in meters (to keep the street and the facades opposite):
MARGIN = 20

In [ ]:
# Celda 2
import os
import requests
import numpy as np
import laspy
import geopandas as gpd

os.makedirs(RAW_DIR, exist_ok=True)

In [ ]:
# Celda 3
# Download each sheet, unless it is already on disk:
for sheet in SHEETS:
    url = BASE_URL + sheet + ".laz"
    path = RAW_DIR + "/" + sheet + ".laz"
    if os.path.exists(path):
        print("Already downloaded:", path)
        continue
    with requests.get(url, stream=True, timeout=600) as answer:
        answer.raise_for_status()
        with open(path, "wb") as f:
            for chunk in answer.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    print("Downloaded:", path, round(os.path.getsize(path) / 1e6, 1), "MB")

In [ ]:
# Celda 4
# Look at the header of one sheet: number of points, CRS, extent:
with laspy.open(RAW_DIR + "/" + SHEETS[0] + ".laz") as reader:
    header = reader.header
    print("Points:", header.point_count)
    print("Version:", header.version, "| point format:", header.point_format.id)
    print("Min:", header.mins)
    print("Max:", header.maxs)
    print("CRS:", header.parse_crs())

In [ ]:
# Celda 5
# Bounding box of the block (all parcels) plus the margin:
parcels = gpd.read_file(GPKG, layer="parcelas")
xmin, ymin, xmax, ymax = parcels.total_bounds
xmin = xmin - MARGIN
ymin = ymin - MARGIN
xmax = xmax + MARGIN
ymax = ymax + MARGIN
print("Clip box:", round(xmin, 1), round(ymin, 1), round(xmax, 1), round(ymax, 1))

In [ ]:
# Celda 6
# Read each sheet in chunks and keep only the points inside the clip box:
# (a whole sheet has about 100 million points, too many to load at once)
# We keep real coordinates (meters), because each sheet stores them with its own offset:
# sheet 439-4474 -> offset x = 439000, sheet 440-4474 -> offset x = 440000
fields = ["x", "y", "z", "intensity", "classification", "red", "green", "blue"]
kept = {}
for name in fields:
    kept[name] = []

for sheet in SHEETS:
    with laspy.open(RAW_DIR + "/" + sheet + ".laz") as reader:
        for points in reader.chunk_iterator(5_000_000):
            px = np.array(points.x)
            py = np.array(points.y)
            inside = (px >= xmin) & (px <= xmax) & (py >= ymin) & (py <= ymax)
            for name in fields:
                kept[name].append(np.array(points[name])[inside])
    print(sheet, "done")

for name in fields:
    kept[name] = np.concatenate(kept[name])
print(len(kept["x"]), "points kept")

In [ ]:
# Celda 7
# Save the clipped cloud in one LAZ file, with one common header (offset at the clip box corner):
header = laspy.LasHeader(version="1.2", point_format=2)
header.offsets = [np.floor(xmin), np.floor(ymin), 0]
header.scales = [0.001, 0.001, 0.001]
clip = laspy.LasData(header)
for name in fields:
    setattr(clip, name, kept[name])
clip.write(OUT_FILE)
print("Saved:", OUT_FILE, round(os.path.getsize(OUT_FILE) / 1e6, 1), "MB")
print("x from", round(clip.x.min(), 1), "to", round(clip.x.max(), 1))

In [ ]:
# Celda 8
# Points and heights per class:
# 1 unclassified, 2 ground, 3-5 vegetation, 6 building
names = {1: "unclassified", 2: "ground", 3: "low veg.", 4: "medium veg.", 5: "high veg.", 6: "building"}
classes = np.array(clip.classification)
z = np.array(clip.z)
for c in sorted(set(classes)):
    zc = z[classes == c]
    print(c, names.get(c, "?"), len(zc), "points | z min", round(zc.min(), 2), "| z max", round(zc.max(), 2))